## Imports

In [0]:
import requests
import uuid
import json
from datetime import datetime, timezone

In [0]:
exec(open("/Workspace/Users/4020arjumandrashid@gmail.com/NASA-EONET/00_config").read())

In [0]:
%sql
USE CATALOG nasa_eonet;
USE SCHEMA events;

## Helpers

In [0]:
def fetch_data(
    field: str,
    limit: int,
    days: int
):
    url = f"{BASE_URL}/{field}"
    params = {
        "key": API_KEY
    }

    # limit=0 / days=0 mean "use the API default" -> omit the parameter
    # (the API treats an explicit limit=0 as "return zero results")
    if limit > 0:
        params["limit"] = limit
    if days > 0:
        params["days"] = days

    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()

        return {
            "status": "SUCCESS",
            "data": response.json(),
            "error": None
        }
    
    except requests.exceptions.HTTPError as e:
        return {
            "status": "FAILED",
            "data": None,
            "error": f"HTTP Error: {e}"
        }
    
    except requests.exceptions.RequestException as e:
        return {
            "status": "FAILED",
            "data": None,
            "error": f"Request Exception: {e}"
        }

In [0]:
def save_api_data(api_df, entity_type = "cat"):

    try:
        (
            api_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"RAW_{entity_type}_TABLE")
        )

        print("WRITE COMPLETED SUCCESSFULLY")

    except Exception as e:

        print("WRITE FAILED")
        print(type(e).__name__)
        print(str(e))
        raise

## Categories

In [0]:
categories = fetch_data(
    "categories",
    0,
    0
)

In [0]:
categories_df = spark.createDataFrame([
    {
        "id": c["id"],
        "title": c["title"],
        "description": c["description"],
        "ingestion_id": str(uuid.uuid4()),
        "ingestion_timestamp": datetime.now(timezone.utc)
    }
    for c in categories["data"]["categories"]
])

In [0]:
save_api_data(categories_df, "cat")

## Sources

In [0]:
sources = fetch_data(
    "sources",
    0,
    0
)

In [0]:
sources_df = spark.createDataFrame([
    {
        "id": s["id"],
        "title": s["title"],
        "ingestion_id": str(uuid.uuid4()),
        "ingestion_timestamp": datetime.now(timezone.utc)
    }
    for s in sources["data"]["sources"]
])

In [0]:
save_api_data(sources_df, "src")

## Events

In [0]:
events = fetch_data(
    "events",
    0, 
    365
)

In [0]:
events['data']['events'][2]

In [0]:
events_df = spark.createDataFrame([
    {
        'id': e['id'],
        'title': e['title'],
        'description': e['description'],
        'categories': e['categories'],
        'sources': e['sources'],
        'geometries': json.dumps(e['geometries'])
    }
    for e in events['data']['events']
])

In [0]:
save_api_data(events_df, "events")

In [0]:
%sql
select * from raw_events_table limit 10;